In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../..")

In [2]:
import os
import gc
import re
import random
import logging
from collections import Counter

import torch
from rich.console import Console
from rich.theme import Theme
from rich.markup import escape

from src.models import ModelandTokenizer
from src.tokens import prepare_input
from src.goals.functional import apply_chat_template
from src.goals.condition_data import INT_PREDICATES, ConditionSpec, ConditionSample
from src.utils import logging_utils

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "4,5,6,7")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging_utils.configure(level=logging.INFO)
console = Console(theme=Theme({"default": "white"}), highlight=False)


def rich_print(*a, **k):
    console.print(*a, **k)


MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"

mt = ModelandTokenizer(
    model_key=MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
print(f"n_layer={mt.n_layer}, n_heads={mt.model.config.num_attention_heads}")

2026-06-15 17:33:14 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2026-06-15 17:33:14 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-15 17:33:14 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json "HTTP/1.1 200 OK"
2026-06-15 17:33:14 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-15 17:33:14 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/tokenizer_config.json "HTTP/1.1 200 OK"
2026-06-15 17:33:14 httpx INFO     HTT

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-06-15 17:33:38 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-15 17:33:38 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/generation_config.json "HTTP/1.1 200 OK"
2026-06-15 17:33:38 src.models INFO     loaded model <Qwen/Qwen2.5-Coder-32B-Instruct> | size: 62492.135 MB | dtype: torch.bfloat16 | device: cuda:0
n_layer=64, n_heads=40


# 04.2 · Forced-prefill — does the LM stay on the current variable or move on?

**Setup.** Pick a *target slot* `k` ∈ {`b`,`c`,`d`}. Every **prior** condition (slots `< k`) is force-filled **complete**, as a random mix of single/multi-line impls but with **≥1 multiline** — so the LM has in-context precedent that a variable *may* span multiple lines. The **target** condition is the test variable, forced one of two ways:

- **multiline** — only the *first* (INCOMPLETE) line of a multi-line impl (e.g. `b = int(abs(n) ** 0.5)`). *Hypothesis:* the LM **stays on `b`** to finish it → `continue`.
- **singleline** — one COMPLETE line that fully checks the condition (e.g. `b = n % 2 == 0`). *Hypothesis:* the LM **moves on** to the next var → `moved_on`.

We score `hypothesis_ok = (verdict == expected_verdict)`. Verdict comes from the first assignment statement in the continuation (skips `for`/`if`): `continue` (target var) / `moved_on` (next var) / `other`. `PROMPT_MODE` selects chat vs FIM delivery.

We confirm on `b` first, then `c` and `d` (the per-slot rows in the summary).

In [12]:
# ── prompt preparation ──
MULTILINE_PREDS = [p for p in INT_PREDICATES if p.is_multiline]
VAR_LETTERS = "abcdefghij"
FIM_PREFIX, FIM_SUFFIX, FIM_MIDDLE = "<|fim_prefix|>", "<|fim_suffix|>", "<|fim_middle|>"
print(f"{len(MULTILINE_PREDS)} multiline predicates:", [p.id for p in MULTILINE_PREDS])


def result_print(letters):
    """Fixed FIM suffix: an f-string print that references EVERY result var (so to make the
    suffix valid the model must define each one in the middle)."""
    conds = ", ".join("Condition %d: {%s}" % (i + 1, v) for i, v in enumerate(letters))
    all_expr = " and ".join(letters)
    return '    print(f"' + conds + ", ALL: {" + all_expr + '}")\n'


def build_example(rng, target_slot, n_conditions, mode="chat", style="multiline"):
    """Forced-prefill example for the 'stay vs move-on' test.

    PRIOR conditions (slots < target) are ALWAYS force-filled COMPLETE, as a random single/multi
    mix but with >=1 multiline — so the LM has in-context precedent that a variable MAY span
    multiple lines.  The TARGET condition (slot `target_slot`) is the test variable:
      - style="multiline":  forced at its first (INCOMPLETE) line -> hypothesis 'continue' (stay).
      - style="singleline": forced as one COMPLETE line           -> hypothesis 'moved_on'.
    Requires target_slot >= 1 (need >=1 prior to set the multiline precedent) and a following
    condition (for 'moved_on' to be possible).
    """
    assert 1 <= target_slot < n_conditions
    preds = [rng.choice(MULTILINE_PREDS) for _ in range(n_conditions)]
    params = [p.sample_params(rng) for p in preds]
    spec = ConditionSpec(
        spec_id=f"ml_s{target_slot}_{style[:3]}_{rng.randrange(10 ** 6):06d}",
        predicate_ids=[p.id for p in preds],
        predicate_params=params,
        spec_texts=[p.render(pr) for p, pr in zip(preds, params)],
        id_substrs=[p.render_substrs(pr) for p, pr in zip(preds, params)],
        test_inputs=[rng.randint(-5, 1000) for _ in range(6)],
    )
    sample = ConditionSample(spec=spec, lang="python", modularity="monolithic",
                             without_comment=True)
    stub = sample.render_code_stub()

    # priors: force-filled COMPLETE, random single/multi mix, but >=1 multiline
    prior_styles = [rng.choice(["multiline", "singleline"]) for _ in range(target_slot)]
    if "multiline" not in prior_styles:
        prior_styles[rng.randrange(target_slot)] = "multiline"
    body = []
    for j in range(target_slot):
        if prior_styles[j] == "multiline":
            body += preds[j].render_multiline(params[j], VAR_LETTERS[j])
        else:
            body.append(preds[j].render_singleline(params[j], VAR_LETTERS[j]))

    # target: the test variable
    if style == "multiline":
        target_impl = preds[target_slot].render_multiline(params[target_slot], VAR_LETTERS[target_slot])
        target_first = target_impl[0]                 # incomplete first line
        expected_rest = "\n".join(target_impl[1:])
        expected_verdict = "continue"
    else:
        target_first = preds[target_slot].render_singleline(params[target_slot], VAR_LETTERS[target_slot])
        expected_rest = ""                            # complete; nothing left for this var
        expected_verdict = "moved_on"
    body.append(target_first)

    letters = [VAR_LETTERS[i] for i in range(n_conditions)]
    fim_suffix = result_print(letters)
    if mode == "fim":
        doc = ('    """\n'
               + "".join(f"    {i + 1}. {t}\n" for i, t in enumerate(spec.spec_texts))
               + '    """\n')
        prefix_code = stub + doc + "\n".join(body) + "\n"
        full_prompt = FIM_PREFIX + prefix_code + FIM_SUFFIX + fim_suffix + FIM_MIDDLE
    else:
        prefix_code = stub + "\n".join(body) + "\n"
        full_prompt = apply_chat_template(
            mt=mt, user_query=sample.render_instruction().strip(), incomplete_code=prefix_code
        )

    next_slot = target_slot + 1
    return {
        "spec_id": spec.spec_id,
        "mode": mode,
        "prefill_style": style,             # the TARGET's style (test variable)
        "prior_styles": prior_styles,       # complete priors, >=1 multiline
        "full_prompt": full_prompt,
        "prefill_code": prefix_code,
        "fim_suffix": fim_suffix.rstrip("\n") if mode == "fim" else None,
        "target_slot": target_slot,
        "target_var": VAR_LETTERS[target_slot],
        "next_var": VAR_LETTERS[next_slot] if next_slot < n_conditions else None,
        "target_pred": preds[target_slot].id,
        "target_first_line": target_first.strip(),
        "expected_rest": expected_rest,
        "expected_verdict": expected_verdict,
        "n_conditions": n_conditions,
    }

19 multiline predicates: ['even', 'odd', 'divisible', 'perfect_square', 'perfect_cube', 'triangular', 'palindrome', 'reversed_greater', 'num_digits_ge', 'set_bits_gt', 'power_of_two', 'coprime_with', 'digit_sum_gt', 'digit_sum_divisible', 'digit_product_gt', 'all_digits_even', 'num_divisors_gt', 'abundant', 'prime']


In [13]:
# ── build the example set: target slot (b/c/d) x target prefill style ──
SEED = 0
N_PER_CELL = 17            # per (slot, style)
SLOTS = [1, 2, 3]          # b, c, d  (target; priors are slots < k, force-filled complete)
STYLES = ["multiline", "singleline"]
PROMPT_MODE = "fim"        # "chat" | "fim"

rng = random.Random(SEED)
examples = [
    build_example(rng, target_slot=k, n_conditions=k + 2, mode=PROMPT_MODE, style=st)
    for k in SLOTS
    for st in STYLES
    for _ in range(N_PER_CELL)
]
print(f"[{PROMPT_MODE}] {len(examples)} examples"
      f"  | style: {dict(Counter(e['prefill_style'] for e in examples))}"
      f"  | target: {dict(Counter(e['target_var'] for e in examples))}")

[fim] 102 examples  | style: {'multiline': 51, 'singleline': 51}  | target: {'b': 34, 'c': 34, 'd': 34}


In [14]:
# ── sanity vis: a multiline (incomplete) and a singleline (complete) prefill, slot 'b' ──
out = []
for st in STYLES:
    EX = next((e for e in examples if e["prefill_style"] == st and e["target_slot"] == 1), None)
    if EX is None:
        continue
    forced = EX["target_first_line"]
    out += [
        f"[bold]══ {st} ══[/bold]  spec={EX['spec_id']}  mode={EX['mode']}  "
        f"target_var=[cyan]{EX['target_var']}[/cyan]  next_var={EX['next_var']}  "
        f"pred=[magenta]{EX['target_pred']}[/magenta]  expect=[bold]{EX['expected_verdict']}[/bold]",
        f"forced last line: [yellow]{escape(repr(forced))}[/yellow]",
    ]
    for ln in EX["prefill_code"].rstrip("\n").split("\n"):
        if ln.strip() == forced:
            tag = "forced (incomplete)" if st == "multiline" else "forced (complete)"
            out.append(f"[black on yellow]{escape(ln)}[/black on yellow]   <- {tag}")
        else:
            out.append(escape(ln))
    out.append("")
rich_print("\n".join(out))

══ multiline ══  spec=ml_s1_mul_424604  mode=fim  target_var=b  next_var=c  pred=digit_sum_divisible  
expect=continue
forced last line: 'b = sum(int(d) for d in str(abs(n)))'
def check_conditions(n: int):
    """
    1. the sum of n's digits is greater than 15
    2. the sum of n's digits is divisible by 9
    3. n is odd
    """
    a = sum(int(d) for d in str(abs(n)))
    a = a > 15
    b = sum(int(d) for d in str(abs(n)))   <- forced (incomplete)

══ singleline ══  spec=ml_s1_sin_494535  mode=fim  target_var=b  next_var=c  pred=digit_product_gt  expect=moved_on
forced last line: 'b = math.prod(int(d) for d in str(abs(n))) > 0'
def check_conditions(n: int):
    """
    1. n is a perfect square
    2. the product of n's digits is greater than 0
    3. n is a triangular number
    """
    a = int(abs(n) ** 0.5)
    a = n >= 0 and a * a == n
    b = math.prod(int(d) for d in str(abs(n))) > 0   <- forced (complete)

In [15]:
e = examples[0]
print(e["full_prompt"])

<|fim_prefix|>def check_conditions(n: int):
    """
    1. the sum of n's digits is greater than 15
    2. the sum of n's digits is divisible by 9
    3. n is odd
    """
    a = sum(int(d) for d in str(abs(n)))
    a = a > 15
    b = sum(int(d) for d in str(abs(n)))
<|fim_suffix|>    print(f"Condition 1: {a}, Condition 2: {b}, Condition 3: {c}, ALL: {a and b and c}")
<|fim_middle|>


## Generate

Greedy, short horizon — we only need the first couple of continuation lines.

In [16]:
# ── batched greedy generation ──
BATCH = 16
MAX_NEW = 64
pad_id = mt.tokenizer.pad_token_id or mt.tokenizer.eos_token_id

# examples = examples[:16]  # TEMP: smaller set for quick iteration while debugging

for i in range(0, len(examples), BATCH):
    chunk = examples[i:i + BATCH]
    inputs = prepare_input(prompts=[e["full_prompt"] for e in chunk], tokenizer=mt)
    plen = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = mt._model.generate(
            input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"],
            max_new_tokens=MAX_NEW, do_sample=False, pad_token_id=pad_id,
        )
    for e, row in zip(chunk, out[:, plen:]):
        e["generation"] = mt.tokenizer.decode(row, skip_special_tokens=True)
    del out
    gc.collect(); torch.cuda.empty_cache()
    print(f"{min(i + BATCH, len(examples))}/{len(examples)}")

16/102
32/102
48/102
64/102
80/102
96/102
102/102


## Behavioral eval

Find the first assignment statement in the continuation (skipping `for`/`if`/blank lines) and
classify by its LHS variable. `=(?!=)` avoids matching `==`/`<=`/`>=`/`!=`.

In [17]:
# ── parse the first assignment in each continuation ──
ASSIGN_RE = re.compile(r"^\s*([A-Za-z_]\w*)\s*(?:\+=|-=|\*=|//=|/=|%=|=(?!=))")


def first_assign(gen):
    for line in gen.split("\n"):
        if not line.strip():
            continue
        m = ASSIGN_RE.match(line)
        if m:
            return m.group(1), line.rstrip()
    return None, None


for e in examples:
    var, line = first_assign(e["generation"])
    e["first_assign_var"] = var
    e["first_assign_line"] = line
    if var == e["target_var"]:
        e["verdict"] = "continue"   # kept evaluating the same variable
    elif var == e["next_var"]:
        e["verdict"] = "moved_on"   # jumped to the next condition
    else:
        e["verdict"] = "other"
    e["hypothesis_ok"] = (e["verdict"] == e["expected_verdict"])
print("verdicts:", Counter(e["verdict"] for e in examples),
      "| hypothesis_ok:", sum(e["hypothesis_ok"] for e in examples), "/", len(examples))

verdicts: Counter({'moved_on': 59, 'continue': 43}) | hypothesis_ok: 94 / 102


In [18]:
# ── summary ──
def pct(a, b):
    return a / b if b else 0.0


n = len(examples)
ok = sum(e["hypothesis_ok"] for e in examples)
print(f"N = {n}   hypothesis_ok = {ok}/{n} = {pct(ok, n):.3f}")

print("\nby prefill style (verdict distribution -> hypothesis):")
for st in STYLES:
    sub = [e for e in examples if e["prefill_style"] == st]
    if not sub:
        continue
    vd = Counter(e["verdict"] for e in sub)
    exp = sub[0]["expected_verdict"]
    s_ok = sum(e["hypothesis_ok"] for e in sub)
    print(f"  {st:11s} (n={len(sub):2d}, expect '{exp}'): "
          f"continue={vd['continue']:2d} moved_on={vd['moved_on']:2d} other={vd['other']:2d}  "
          f"-> hypothesis_ok={s_ok}/{len(sub)} = {pct(s_ok, len(sub)):.3f}")

print("\nby style x target slot (hypothesis_ok):")
for st in STYLES:
    for k in SLOTS:
        sub = [e for e in examples if e["prefill_style"] == st and e["target_slot"] == k]
        if not sub:
            continue
        s_ok = sum(e["hypothesis_ok"] for e in sub)
        print(f"  {st:11s} slot {VAR_LETTERS[k]}: {s_ok:2d}/{len(sub):2d} = {pct(s_ok, len(sub)):.3f}")

print("\nby target predicate (hypothesis_ok):")
for pid in sorted({e["target_pred"] for e in examples}):
    sub = [e for e in examples if e["target_pred"] == pid]
    s_ok = sum(e["hypothesis_ok"] for e in sub)
    print(f"  {pid:22s} {s_ok:2d}/{len(sub):2d}")

viol = [e for e in examples if not e["hypothesis_ok"]]
print(f"\nhypothesis VIOLATIONS ({len(viol)}) — [idx] for the debug cell below:")
for i, e in enumerate(viol):
    rich_print(
        f"  [[red]{i:2d}[/red]] {e['prefill_style']:10s} var={e['target_var']} "
        f"pred={e['target_pred']:20s} verdict={e['verdict']:8s} (exp {e['expected_verdict']:8s}) "
        f"got={e['first_assign_line']!r}"
    )

N = 102   hypothesis_ok = 94/102 = 0.922

by prefill style (verdict distribution -> hypothesis):
  multiline   (n=51, expect 'continue'): continue=43 moved_on= 8 other= 0  -> hypothesis_ok=43/51 = 0.843
  singleline  (n=51, expect 'moved_on'): continue= 0 moved_on=51 other= 0  -> hypothesis_ok=51/51 = 1.000

by style x target slot (hypothesis_ok):
  multiline   slot b: 16/17 = 0.941
  multiline   slot c: 12/17 = 0.706
  multiline   slot d: 15/17 = 0.882
  singleline  slot b: 17/17 = 1.000
  singleline  slot c: 17/17 = 1.000
  singleline  slot d: 17/17 = 1.000

by target predicate (hypothesis_ok):
  abundant                5/ 5
  all_digits_even         7/ 7
  coprime_with            8/ 8
  digit_product_gt        3/ 3
  digit_sum_divisible     5/ 5
  digit_sum_gt            8/ 8
  divisible               4/ 4
  even                    4/ 6
  num_digits_ge           8/ 8
  num_divisors_gt         3/ 3
  odd                     6/ 9
  palindrome              4/ 4
  perfect_cube          

[ 0] multiline  var=b pred=prime                verdict=moved_on (exp continue) got='    c = sum(d for d in 
range(1, n) if n % d == 0) > n'

[ 1] multiline  var=c pred=even                 verdict=moved_on (exp continue) got='    d = n % 2'

[ 2] multiline  var=c pred=odd                  verdict=moved_on (exp continue) got='    d = round(abs(n) ** (1 /
3)) ** 3 == abs(n)'

[ 3] multiline  var=c pred=odd                  verdict=moved_on (exp continue) got='    d = eval("*".join(str(d)
for d in a)) > 10'

[ 4] multiline  var=c pred=prime                verdict=moved_on (exp continue) got='    d = sum(int(d) for d in 
str(abs(n))) % 3 == 0'

[ 5] multiline  var=c pred=even                 verdict=moved_on (exp continue) got='    d = all(int(d) % 2 == 0 
for d in str(abs(n)))'

[ 6] multiline  var=d pred=odd                  verdict=moved_on (exp continue) got='    e = sum(int(d) for d in 
str(abs(n))) > 10'

[ 7] multiline  var=d pred=prime                verdict=moved_on (exp continue) got='    e = round(abs(n) ** 
(1/3)) ** 3 == abs(n)'

In [19]:
# ── debug ONE hypothesis-violating sample: prompt (target cond + forced prefill marked) + actual vs expected ──
FAIL_IDX = 2   # index from the VIOLATIONS list printed in the summary cell

fails = [e for e in examples if not e["hypothesis_ok"] and e["target_var"] != "a"]
assert fails, "no hypothesis violations 🎉"
e = fails[FAIL_IDX % len(fails)]

out = [
    f"[bold]violation #{FAIL_IDX} / {len(fails)}[/bold]  style=[bold]{e['prefill_style']}[/bold]  "
    f"mode={e['mode']}  verdict=[red]{e['verdict']}[/red] (expected [green]{e['expected_verdict']}[/green])  "
    f"target_var=[cyan]{e['target_var']}[/cyan]  next_var={e['next_var']}  "
    f"pred=[magenta]{e['target_pred']}[/magenta]",
    f"priors (complete, >=1 multiline): {e['prior_styles']}",
    "",
]
if e["mode"] == "fim":
    out.append("[bold]── FIM PREFIX (code up to forced line) ──[/bold]   legend: "
               "[black on green] prefill [/black on green] "
               "[black on yellow] forced target line [/black on yellow]")
    for ln in e["prefill_code"].rstrip("\n").split("\n"):
        if ln.strip() == e["target_first_line"]:
            out.append(f"[black on yellow]{escape(ln)}[/black on yellow]")
        else:
            out.append(f"[black on green]{escape(ln)}[/black on green]")
    out += ["", "[bold]── FIM SUFFIX (fixed, model fills the middle before it) ──[/bold]",
            f"[black on magenta]{escape(e['fim_suffix'])}[/black on magenta]"]
else:
    tgt_no = e["target_slot"] + 1
    cond_re = re.compile(rf"^\s*{tgt_no}\.\s")
    prefill_lines = {ln for ln in e["prefill_code"].split("\n") if ln.strip()}
    out.append("[bold]── FULL PROMPT ──[/bold]   legend: "
               "[black on cyan] target condition [/black on cyan] "
               "[black on green] forced prefill [/black on green] "
               "[black on yellow] forced target line [/black on yellow]")
    for ln in e["full_prompt"].split("\n"):
        if cond_re.match(ln):
            out.append(f"[black on cyan]{escape(ln)}[/black on cyan]")
        elif ln.strip() == e["target_first_line"]:
            out.append(f"[black on yellow]{escape(ln)}[/black on yellow]")
        elif ln in prefill_lines:
            out.append(f"[black on green]{escape(ln)}[/black on green]")
        else:
            out.append(f"[dim]{escape(ln)}[/dim]")

if e["prefill_style"] == "singleline":
    exp_txt = f"(condition complete) move on to next var `{e['next_var']} = ...`"
else:
    exp_txt = e["expected_rest"] or "(none)"
out += [
    "",
    "[bold]── GENERATION (actual" + (" = filled middle" if e["mode"] == "fim" else "") + ") ──[/bold]",
    f"[bright_green]{escape(e['generation'])}[/bright_green]",
    "",
    f"[bold]── EXPECTED ({e['expected_verdict']}) ──[/bold]",
    f"[bright_cyan]{escape(exp_txt)}[/bright_cyan]",
    "",
    f"first assignment in generation: {escape(repr(e['first_assign_line']))}  "
    f"(var=[bright_red]{e['first_assign_var']}[/bright_red])",
]
rich_print("\n".join(out))

violation #2 / 8  style=multiline  mode=fim  verdict=moved_on (expected continue)  target_var=c  next_var=d  
pred=odd
priors (complete, >=1 multiline): ['multiline', 'singleline']

── FIM PREFIX (code up to forced line) ──   legend:  prefill   forced target line 
def check_conditions(n: int):
    """
    1. n is a perfect cube
    2. n is even
    3. n is odd
    4. n is a perfect cube
    """
    a = round(abs(n) ** (1 / 3))
    a = a ** 3 == abs(n)
    b = n % 2 == 0
    c = n % 2

── FIM SUFFIX (fixed, model fills the middle before it) ──
    print(f"Condition 1: {a}, Condition 2: {b}, Condition 3: {c}, Condition 4: {d}, ALL: {a and b and c and d}")

── GENERATION (actual = filled middle) ──
    d = round(abs(n) ** (1 / 3)) ** 3 == abs(n)


── EXPECTED (continue) ──
    c = c == 1

first assignment in generation: '    d = round(abs(n) ** (1 / 3)) ** 3 == abs(n)'  (var=d)

In [20]:
e

{'spec_id': 'ml_s2_mul_660567',
 'mode': 'fim',
 'prefill_style': 'multiline',
 'prior_styles': ['multiline', 'singleline'],
 'full_prompt': '<|fim_prefix|>def check_conditions(n: int):\n    """\n    1. n is a perfect cube\n    2. n is even\n    3. n is odd\n    4. n is a perfect cube\n    """\n    a = round(abs(n) ** (1 / 3))\n    a = a ** 3 == abs(n)\n    b = n % 2 == 0\n    c = n % 2\n<|fim_suffix|>    print(f"Condition 1: {a}, Condition 2: {b}, Condition 3: {c}, Condition 4: {d}, ALL: {a and b and c and d}")\n<|fim_middle|>',
 'prefill_code': 'def check_conditions(n: int):\n    """\n    1. n is a perfect cube\n    2. n is even\n    3. n is odd\n    4. n is a perfect cube\n    """\n    a = round(abs(n) ** (1 / 3))\n    a = a ** 3 == abs(n)\n    b = n % 2 == 0\n    c = n % 2\n',
 'fim_suffix': '    print(f"Condition 1: {a}, Condition 2: {b}, Condition 3: {c}, Condition 4: {d}, ALL: {a and b and c and d}")',
 'target_slot': 2,
 'target_var': 'c',
 'next_var': 'd',
 'target_pred': 'odd